In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.pricecharting.com/console/pokemon-mega-evolution"
resp = requests.get(url)
print("Código de respuesta HTTP:", resp.status_code)



Código de respuesta HTTP: 200


In [ ]:
# Creamos el objeto BeautifulSoup para interpretar el HTML
soup = BeautifulSoup(resp.text, "html.parser")

# Buscamos la tabla con id="games_table"
tabla = soup.find("table", id="games_table")

# Comprobamos si la tabla se ha encontrado correctamente
print("¿Tabla encontrada?:", tabla is not None)


¿Tabla encontrada?: True


In [ ]:
# 1) Obtenemos todas las filas (<tr>) del cuerpo de la tabla
filas = tabla.tbody.find_all("tr")

# 2) Vemos cuántas filas/cartos hay en este set
print("Número de filas (cartas) encontradas:", len(filas))

# 3) Cogemos la primera fila de la tabla
primera_fila = filas[0]

# 4) Mostramos el HTML completo de esa primera fila de forma legible
print(primera_fila.prettify())


Número de filas (cartas) encontradas: 50
<tr data-product="10669978" id="product-10669978">
 <td class="image">
  <div>
   <a href="https://www.pricecharting.com/game/pokemon-mega-evolution/mega-lucario-ex-188" title="10669978">
    <img class="photo" loading="lazy" src="https://storage.googleapis.com/images.pricecharting.com/hgvcsnv7zu4qfapt/60.jpg">
    </img>
   </a>
  </div>
 </td>
 <td class="title" title="10669978">
  <a href="/game/pokemon-mega-evolution/mega-lucario-ex-188">
   Mega Lucario Ex #188
  </a>
 </td>
 <td class="price numeric used_price">
  <span class="js-price">
   $352.67
  </span>
 </td>
 <td class="price numeric cib_price">
  <span class="js-price">
   $422.28
  </span>
 </td>
 <td class="price numeric new_price">
  <span class="js-price">
   $1,043.56
  </span>
 </td>
 <td class="add_to container">
  <ul class="add_to collection">
   <li class="add_to collection">
    <ul class="list" data-product-id="10669978">
     <li class="list-item">
      <a class="js-a

In [ ]:
# EXTRAER DE LA PRIMERA FILA LOS 4 DATOS IMPORTANTES

# Nombre
nombre = primera_fila.find("td", class_="title").get_text(strip=True)

# Ungraded
ungraded = primera_fila.find("td", class_="used_price").get_text(strip=True)

# Grade 9
grade9 = primera_fila.find("td", class_="cib_price").get_text(strip=True)

# PSA 10
psa10 = primera_fila.find("td", class_="new_price").get_text(strip=True)

print("Nombre  :", nombre)
print("Ungraded:", ungraded)
print("Grade 9 :", grade9)
print("PSA 10  :", psa10)


Nombre  : Mega Lucario Ex #188
Ungraded: $352.67
Grade 9 : $422.28
PSA 10  : $1,043.56


In [ ]:
import pandas as pd  # por si aún no lo has importado

# Función auxiliar para no rompernos si falta alguna celda
def extraer_celda(fila, class_name):
    celda = fila.find("td", class_=class_name)
    if celda is not None:
        return celda.get_text(strip=True)
    else:
        return None

# Lista donde guardaremos todas las cartas
cartas = []

# Recorremos TODAS las filas de la tabla
for fila in filas:
    nombre = extraer_celda(fila, "title")
    ungraded = extraer_celda(fila, "used_price")
    grade9 = extraer_celda(fila, "cib_price")
    psa10 = extraer_celda(fila, "new_price")

    # Solo añadimos la fila si tiene nombre (para evitar filas raras/vacías)
    if nombre is not None:
        cartas.append({
            "nombre": nombre,
            "ungraded": ungraded,
            "grade9": grade9,
            "psa10": psa10
        })

# Convertimos la lista de diccionarios en un DataFrame
df = pd.DataFrame(cartas)

# Vemos las primeras filas para comprobar
df.head()


,nombre,ungraded,grade9,psa10
0,Mega Lucario Ex #188,$352.67,$422.28,"$1,043.56"
1,Mega Lucario Ex #179,$212.51,$319.00,$516.01
2,Mega Gardevoir ex #178,$193.65,$269.50,$499.17
3,Mega Latias ex #181,$125.00,$207.50,$415.00
4,Mega Gardevoir Ex #187,$308.59,$293.01,$737.44


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
ruta_csv = '/content/drive/MyDrive/pokemon_mega_evolution.csv'
df.to_csv(ruta_csv, index=False)

print("CSV guardado en:", ruta_csv)


CSV guardado en: /content/drive/MyDrive/pokemon_mega_evolution.csv


In [ ]:
url_cat = "https://www.pricecharting.com/category/pokemon-cards"

resp_cat = requests.get(url_cat)
print("Código de respuesta categoría:", resp_cat.status_code)

soup_cat = BeautifulSoup(resp_cat.text, "html.parser")


Código de respuesta categoría: 200


In [ ]:
# Buscamos todos los enlaces <a> dentro de la página de categorías
enlaces = soup_cat.find_all("a")

urls_sets = []

for a in enlaces:
    href = a.get("href", "")
    # Nos interesan solo los enlaces que empiezan por "/console/pokemon"
    if href.startswith("/console/pokemon"):
        url_completa = "https://www.pricecharting.com" + href
        if url_completa not in urls_sets:
            urls_sets.append(url_completa)

# Vemos cuántos sets hemos encontrado y los primeros 10
print("Número de sets encontrados:", len(urls_sets))
urls_sets[:10]


Número de sets encontrados: 302


['https://www.pricecharting.com/console/pokemon-promo',
 'https://www.pricecharting.com/console/pokemon-mega-evolution',
 'https://www.pricecharting.com/console/pokemon-prismatic-evolutions',
 'https://www.pricecharting.com/console/pokemon-destined-rivals',
 'https://www.pricecharting.com/console/pokemon-scarlet-&-violet-151',
 'https://www.pricecharting.com/console/pokemon-base-set',
 'https://www.pricecharting.com/console/pokemon-surging-sparks',
 'https://www.pricecharting.com/console/pokemon-crown-zenith',
 'https://www.pricecharting.com/console/pokemon-celebrations',
 'https://www.pricecharting.com/console/pokemon-japanese-promo']

#Funcion para scrapear sets

In [ ]:
def scrapear_set(url_set):
    """
    Descarga un set de PriceCharting (URL /console/...)
    y devuelve una lista de diccionarios con las cartas de ese set.
    """
    print("Scrapeando set:", url_set)

    resp = requests.get(url_set)
    if resp.status_code != 200:
        print("  Error HTTP:", resp.status_code)
        return []  # devolvemos lista vacía si hay fallo

    soup = BeautifulSoup(resp.text, "html.parser")

    tabla = soup.find("table", id="games_table")
    if tabla is None:
        print("  No se encontró tabla en este set.")
        return []

    filas = tabla.tbody.find_all("tr")

    cartas_set = []

    for fila in filas:
        nombre   = extraer_celda(fila, "title")
        ungraded = extraer_celda(fila, "used_price")
        grade9   = extraer_celda(fila, "cib_price")
        psa10    = extraer_celda(fila, "new_price")

        if nombre is not None:
            cartas_set.append({
                "nombre": nombre,
                "ungraded": ungraded,
                "grade9": grade9,
                "psa10": psa10
            })

    print("  Cartas en este set:", len(cartas_set))
    return cartas_set


In [ ]:
lista_cartas = scrapear_set("https://www.pricecharting.com/console/pokemon-mega-evolution")


Scrapeando set: https://www.pricecharting.com/console/pokemon-mega-evolution
  Cartas en este set: 50


In [ ]:
todas_las_cartas = []

In [ ]:
import time
from tqdm import tqdm


print("Iniciando scraping…")

for url_set in tqdm(urls_sets):
    try:
        cartas_set = scrapear_set(url_set)
        todas_las_cartas.extend(cartas_set)

        # Pausa para no saturar al servidor
        time.sleep(0.5)

    except Exception as e:
        print("Error en:", url_set)
        print("Detalle del error:", e)



Iniciando scraping…


  0%|          | 0/302 [00:00<?, ?it/s]

Scrapeando set: https://www.pricecharting.com/console/pokemon-promo
  Cartas en este set: 50


  0%|          | 1/302 [00:01<06:07,  1.22s/it]

Scrapeando set: https://www.pricecharting.com/console/pokemon-mega-evolution
  Cartas en este set: 50


...

 99%|█████████▉| 300/302 [05:05<00:01,  1.03it/s]

Scrapeando set: https://www.pricecharting.com/console/pokemon-world-championships-2023
  Cartas en este set: 50


100%|█████████▉| 301/302 [05:06<00:00,  1.06it/s]

Scrapeando set: https://www.pricecharting.com/console/pokemon-xy
  Cartas en este set: 50


100%|██████████| 302/302 [05:07<00:00,  1.02s/it]


# Crear dataframe

In [ ]:
df_total = pd.DataFrame(todas_las_cartas)
df_total.head()
print(len(df_total))


NameError: name 'todas_las_cartas' is not defined

In [ ]:
ruta_csv = '/content/drive/MyDrive/todas_las_cartas_pokemon.csv'
df_total.to_csv(ruta_csv, index=False)

print("CSV guardado en:", ruta_csv)


NameError: name 'df_total' is not defined

In [ ]:


ruta = "/content/drive/MyDrive/Python/Cartas_Pokemon/Lista_pokemon_oficial_limpia.csv"

df_lista_total = pd.read_csv(ruta, sep=";", decimal=",")
df.head()



,nombre,ungraded,grade9,psa10
0,Mega Lucario Ex #188,$352.67,$422.28,"$1,043.56"
1,Mega Lucario Ex #179,$212.51,$319.00,$516.01
2,Mega Gardevoir ex #178,$193.65,$269.50,$499.17
3,Mega Latias ex #181,$125.00,$207.50,$415.00
4,Mega Gardevoir Ex #187,$308.59,$293.01,$737.44


In [ ]:
filtro_ungraded = df_lista_total[df_lista_total["ungraded"] < 50]
filtro_ungraded.head()


,nombre,ungraded,grade9,psa10,Beneficio
3,Charizard VStar #SWSH262,49.46,79.00,782.50,733.04
4,Eevee #173,12.23,19.50,90.63,78.40
6,Charmander #44,32.97,65.00,325.05,292.08
7,Charizard VMax #SWSH261,32.50,64.11,490.00,457.50
8,Lucario VSTAR #SWSH291,11.00,21.25,111.78,100.78


In [ ]:
len(filtro_ungraded)

11134

In [ ]:
filtro_beneficio = filtro_ungraded[filtro_ungraded["Beneficio"] > 100]
filtro_beneficio.head()

,nombre,ungraded,grade9,psa10,Beneficio
3,Charizard VStar #SWSH262,49.46,79.00,782.50,733.04
6,Charmander #44,32.97,65.00,325.05,292.08
7,Charizard VMax #SWSH261,32.50,64.11,490.00,457.50
8,Lucario VSTAR #SWSH291,11.00,21.25,111.78,100.78
9,Snorlax #51,13.85,29.96,160.36,146.51


In [ ]:
len(filtro_beneficio)

3360

In [ ]:
ruta_csv = '/content/drive/MyDrive/Python/Cartas_Pokemon/todas_las_cartas_pokemon_filtradas.csv'
filtro_beneficio.to_csv(ruta_csv, index=False)

print("CSV guardado en:", ruta_csv)

CSV guardado en: /content/drive/MyDrive/Python/Cartas_Pokemon/todas_las_cartas_pokemon_filtradas.csv
